# Langchain Messages

This notebook introduces the message-based API in LangChain, which is used to communicate with chat models. Instead of sending only a plain string, we send a structured list of messages with defined roles. This helps the model understand context, instructions, and the flow of a conversation more clearly.

A message is essentially a unit of communication between the user, the assistant, and sometimes tools. In chat-based systems, messages allow the model to remember previous context and respond in a more natural and useful way.

In [30]:
import torch
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"config\.env")

# --- ROCm/CUDA device check ---
# ROCm exposes itself to PyTorch through the same torch.cuda API as NVIDIA CUDA,
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detected: {device_name} ({total_vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected by torch — falling back to CPU. Check your ROCm/torch install.")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

GRmodel = init_chat_model("groq:qwen/qwen3.6-27b",)

GPU detected: AMD Radeon RX 7900 XT (20.0 GB VRAM)


In [31]:
GRmodel.invoke("Please tell what is artificial intelligence?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user asks "Please tell what is artificial intelligence?" This is a straightforward, foundational question about AI.\n\n2.  **Identify Key Components of AI Definition**:\n   - What is AI? (Core definition)\n   - How does it work? (Basic mechanisms)\n   - Key characteristics/capabilities\n   - Types/categories (Narrow vs. General, etc.)\n   - Real-world applications\n   - Brief note on current state/limitations\n   - Keep it clear, concise, and accessible\n\n3.  **Draft - Mental Refinement**:\n   Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks that typically require human cognition. These tasks include problem-solving, recognizing patterns, understanding language, making decisions, and even creating content.\n\n   AI works by using algorithms and large amounts of data to identify patterns, make predi

## Text Prompts

A text prompt is the simplest way to ask a model a question. You provide a single string, and the model responds based on that input.

This is useful for quick questions, example prompts, and basic interactions. For example, asking “What is LangChain?” sends a plain-language instruction to the model, which then generates a response.

Use when:
- you want a quick one-off answer
- the task is simple and does not need much conversation history
- you are testing a model or comparing different prompts

Text prompts are easy to start with, but for more advanced behavior, message-based prompts are often better because they let you provide roles such as system instructions and user questions.

In [32]:
GRmodel.invoke("What is langchain?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query:** The user asks "What is langchain?" This is a straightforward definition/explanation question about a specific technology/framework.\n\n2.  **Identify Key Subject:** LangChain is a popular open-source framework/library for developing applications powered by large language models (LLMs).\n\n3.  **Core Concepts to Cover:**\n   - Definition/Purpose\n   - Key Features/Capabilities\n   - Architecture/Components (Chains, Agents, Memory, Tools, etc.)\n   - Use Cases\n   - Languages/Platforms\n   - Current Status/Support\n   - Brief mention of alternatives/context (optional but helpful)\n\n4.  **Draft - Mental Refinement (Structure):**\n   - **Introduction:** Define LangChain clearly and concisely.\n   - **Core Purpose:** Why it exists (simplify LLM app development, handle complexity).\n   - **Key Components:** Break down the main building blocks (Chains, Agents, Memory, Tools/Integrations, Prompt Templa

## Message Prompts

Message prompts are more structured than plain text prompts. Instead of sending one string, we send a list of role-based messages. This gives the model clearer instructions and a better conversation context.

Types:
- System: instructions that define how the model should behave or what style it should follow.
- Human: the user's input or question.
- AI: previous assistant responses, useful when continuing a conversation.
- Tool: messages generated by tools or external systems when the model interacts with them.

### System Message
A system message sets the overall behavior of the model. It is usually used to tell the assistant how to respond, what tone to use, or what constraints to follow.

### Human Message
A human message represents the user’s actual request. This is the normal question or instruction the model should respond to.

### AI Message
An AI message represents a prior response from the assistant. It is useful when continuing a conversation and maintaining context across multiple turns.

### Tool Message
A tool message contains output from an external tool or function. It helps the model understand results from tools like search, calculators, or APIs when used in a workflow.

In [33]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response = GRmodel.invoke(messages)
response.content

'\n<think>\nHere\'s a thinking thinking sequence\n\n1.  **Deconstruct the Request:**\n    *   Topic: Artificial Intelligence (AI).\n    *   Format: Poem.\n    *   Implicit/Explicit Goals: Capture the essence, potential, fears, and nature of AI. The user didn\'t specify style, tone, or length, so I should aim for a balanced, evocative piece that touches on multiple facets.\n\n2.  **Brainstorming Imagery and Themes:**\n    *   *Keywords:* Code, silicon, neural networks, dreams, electric, learning, mirror, human, future, ghost in the machine, data, algorithms, creativity, cold, warm, understanding.\n    *   *Themes:*\n        *   Creation: Humans building AI (Prometheus, Frankenstein vibes, but maybe more modern).\n        *   Nature: Is it alive? Is it a tool? Is it a mirror?\n        *   Capabilities: Calculation, art, language, speed.\n        *   Risks/Ethics: Loss of humanity, bias, control, singularity.\n        *   Symbiosis: Working together, enhancement.\n\n3.  **Determining Stru

In [34]:
system_message = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_message,
    HumanMessage("How do i create a REST API?"),
]

response = GRmodel.invoke(messages)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query:** The user asks "How do i create a REST API?" This is a broad question, but given my role as a senior Python developer with expertise in web frameworks, I should focus on Python, specifically using a popular, modern framework like FastAPI or Flask. FastAPI is generally recommended nowadays for its performance, automatic documentation, type hints, and ease of use.\n\n2.  **Identify Key Components of a REST API:**\n   - HTTP methods (GET, POST, PUT, DELETE, etc.)\n   - Endpoints/URLs\n   - Request/Response handling (JSON)\n   - Routing\n   - Data validation\n   - Error handling\n   - Documentation (optional but highly recommended)\n\n3.  **Choose the Best Framework for the Example:** FastAPI is the modern standard in Python for REST APIs. It\'s fast, developer-friendly, and handles many best practices out of the box. I\'ll use FastAPI as the primary example, but I\'ll briefly mention Flask/Django REST Framework as al

In [35]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [36]:
response = GRmodel.invoke([
  human_msg
])
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user simply said "Hello!"\n2.  **Identify Intent:** This is a standard greeting. The user is initiating a conversation.\n3.  **Determine Response Goal:** Acknowledge the greeting, be friendly, and invite the user to ask a question or share what\'s on their mind.\n4.  **Formulate Response:** Keep it warm, concise, and open-ended. Example: "Hello! How can I assist you today?" or "Hi there! What\'s on your mind?"\n5.  **Refine Response:** Match the tone of the user (friendly, casual). "Hello! How can I help you today?" works well. Keep it professional yet approachable.\n6.  **Final Output Generation:** Output the refined response.✅\n</think>\n\nHello! How can I help you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 191, 'prompt_tokens': 12, 'total_tokens': 203, 'completion_time': 0.373655196, 'completion_tokens_details': None, 'prompt_time': 0.000588268, 

In [37]:
# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = GRmodel.invoke(messages)
print(response.content)




<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asks "Great! What's 2+2?"
2.  **Identify Core Question:** Simple arithmetic question: 2 + 2 = ?
3.  **Determine Answer:** 2 + 2 = 4
4.  **Formulate Response:** Keep it clear, direct, and friendly. "2 + 2 equals 4!" or just "4".
5.  **Check for Nuance/Traps:** None. It's a straightforward math question.
6.  **Draft Response:** "2 + 2 equals 4! Let me know if you need help with anything else."
7.  **Final Output Generation:** Matches draft.✅
</think>

2 + 2 equals **4**! Let me know if you need help with anything else. 😊


In [38]:
response.usage_metadata

{'input_tokens': 55, 'output_tokens': 187, 'total_tokens': 242}

In [39]:
# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = GRmodel.invoke(messages)  # Model processes the result

In [40]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [41]:
response

AIMessage(content='\n<think>\nThe user asked for the weather in San Francisco.\nThe weather tool returned "Sunny, 72°F".\nI will report this information to the user.\n</think>\n\nIt\'s currently sunny and 72°F in San Francisco.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 69, 'total_tokens': 120, 'completion_time': 0.099915222, 'completion_tokens_details': None, 'prompt_time': 0.005662782, 'prompt_tokens_details': None, 'queue_time': 0.047537659, 'total_time': 0.105578004}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_518cb5990e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a026a2-509e-7551-be55-f8ebbb2d48cf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 51, 'total_tokens': 120})